In [1]:
# rate = log2 k bits
# k = 2^rate = 2^2 = 4
# 4 quantization points

# for 4 quantization points, we need 5 decision boundaries (bins): u0,..,u4
# where u0 = -inf, u4 = +inf

# q_i: reconstruction points - The specific value assigned to any input falling within a bin.

# D = E(X-X^)^2 = int(u0, u4) (x-x^)^2fxdx = sum(k=0, k-1) int(u_k, u_k+1) (x-q_k)^2fxdx

#since k is even: 
# q_i: -3/2 Delta, -1/2 Delta, 1/2 Delta, 3/2 Delta 
# b_i: -inf, Delta, 0, Delta, +inf

In [2]:
import numpy as np
from scipy.integrate import quad
from scipy.optimize import minimize_scalar

## 1. Scalar Uniform Quantization

In [3]:
def laplacian_pdf(x):
    return (1/6) * np.exp(-np.abs(x)/3)

def distortion_uniform_scalar(delta):
    # since we only need to integrate over the positive axis (then mulitply by 2)
    # we need q3, q4, u3
    q3 = 1/2 * delta
    q4 = 3/2 * delta
    u3 = delta

    term1, _ = quad(lambda x: (x-q3)**2 * laplacian_pdf(x), 0, u3)
    term2, _ = quad(lambda x: (x-q4)**2 * laplacian_pdf(x), u3, np.inf)

    return 2 * (term1 + term2)

In [4]:
# minimize distortion
result = minimize_scalar(distortion_uniform_scalar, bounds=(0, 10), method='bounded')

In [5]:
optimal_delta_us = result.x
optimal_distortion_us = result.fun
quantization_points_us = np.array([-3/2 * optimal_delta_us, -1/2 * optimal_delta_us, 1/2 * optimal_delta_us, 3/2 * optimal_delta_us])
decision_boundaries_us = np.array([-np.inf, -optimal_delta_us, 0, optimal_delta_us, +np.inf])

print(f"Optimal Delta: {optimal_delta_us:.4f}")
print(f"Optimal Distortion: {optimal_distortion_us:.4f}")
print(f"Optimal Quantization points: {quantization_points_us}")
print(f"Optimal Decision boundaries: {decision_boundaries_us}")

Optimal Delta: 4.6134
Optimal Distortion: 3.5334
Optimal Quantization points: [-6.92012496 -2.30670832  2.30670832  6.92012496]
Optimal Decision boundaries: [       -inf -4.61341664  0.          4.61341664         inf]


## 2. Lloyd-Max Quantizer

In [20]:
def lloyd_max_quantizer(init_q, max_iterations = 100,):
    q = init_q
    for iter in range(max_iterations):

        decision_boundaries = (q[1:]+q[:-1])/2
        # q_old = q.copy()

        for i in range(len(q)):
            if i == 0:
                numerator,_ = quad(lambda x: x * laplacian_pdf(x),-np.inf,decision_boundaries[i])
                denominator,_ = quad(lambda x: laplacian_pdf(x), -np.inf, decision_boundaries[i])
            elif i == (len(q)-1):
                numerator,_ = quad(lambda x: x * laplacian_pdf(x),decision_boundaries[i-1], np.inf)
                denominator,_ = quad(lambda x: laplacian_pdf(x), decision_boundaries[i-1], np.inf)
            else:
                numerator,_ = quad(lambda x: x * laplacian_pdf(x),decision_boundaries[i-1], decision_boundaries[i])
                denominator,_ = quad(lambda x: laplacian_pdf(x), decision_boundaries[i-1], decision_boundaries[i])
            
            q[i] = numerator/denominator
        
        # if np.linalg.norm(q_old - q) < threshold:
        #     break
    
    return q, (q[1:]+q[:-1])/2

def calculate_distortion_lloyd_max(quantization_points, decision_boundaries):
    distortion = 0
    for i in range(0, len(quantization_points)):
        part_i, _ = quad(lambda x: (x - quantization_points[i])**2 * laplacian_pdf(x), decision_boundaries[i], decision_boundaries[i+1])
        distortion += part_i
        
    return distortion


In [21]:
init_q_lm = np.array([-10,-4,4,10])

optimal_q_lm, decision_boundaries_lm = lloyd_max_quantizer(init_q_lm)
decision_boundaries_lm = np.array([-np.inf] + list(decision_boundaries_lm) + [np.inf])
optimal_distortion_lm = calculate_distortion_lloyd_max(optimal_q_lm, decision_boundaries_lm)

print(f"Optimal Distortion: {optimal_distortion_lm:.4f}")
print(f"Optimal Quantization points: {optimal_q_lm}")
print(f"Optimal Decision boundaries: {decision_boundaries_lm}")

Optimal Distortion: 3.5105
Optimal Quantization points: [-7 -1  1  7]
Optimal Decision boundaries: [-inf  -4.   0.   4.  inf]


In [22]:
init_q_lm = quantization_points_us

optimal_q_lm, decision_boundaries_lm = lloyd_max_quantizer(init_q_lm)
decision_boundaries_lm = np.array([-np.inf] + list(decision_boundaries_lm) + [np.inf])
optimal_distortion_lm = calculate_distortion_lloyd_max(optimal_q_lm, decision_boundaries_lm)

print(f"Optimal Distortion: {optimal_distortion_lm:.4f}")
print(f"Optimal Quantization points: {optimal_q_lm}")
print(f"Optimal Decision boundaries: {decision_boundaries_lm}")

Optimal Distortion: 3.1715
Optimal Quantization points: [-7.78087278 -1.78087278  1.78087278  7.78087278]
Optimal Decision boundaries: [       -inf -4.78087278  0.          4.78087278         inf]


In [23]:
init_q_lm = np.array([-12, -8, 6, 10])

optimal_q_lm, decision_boundaries_lm = lloyd_max_quantizer(init_q_lm)
decision_boundaries_lm = np.array([-np.inf] + list(decision_boundaries_lm) + [np.inf])
optimal_distortion_lm = calculate_distortion_lloyd_max(optimal_q_lm, decision_boundaries_lm)

print(f"Optimal Distortion: {optimal_distortion_lm:.4f}")
print(f"Optimal Quantization points: {optimal_q_lm}")
print(f"Optimal Decision boundaries: {decision_boundaries_lm}")

Optimal Distortion: 3.2900
Optimal Quantization points: [-7 -2  1  7]
Optimal Decision boundaries: [-inf -4.5 -0.5  4.   inf]
